# Notebook to do assignment of classes

In [1]:
ROOT= '../'

In [2]:
!pip install wildlife-datasets git+https://github.com/WildlifeDatasets/wildlife-tools --quiet --upgrade-strategy only-if-needed

In [ ]:
import shutil, os
import warnings
warnings.filterwarnings('ignore')
import os

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

from sklearn.cluster import DBSCAN
from sklearn.metrics import pairwise_distances

Config Parameters

In [4]:
import torch

CFG = dict(
    model_name        = 'vit_base_patch16_224.dino',
    img_size          = 224,
    embed_dim         = 512,      # ViT-Base output dim; set smaller (e.g. 512) to project down
    arcface_s         = 16.0,     # scale
    arcface_m         = 0.35,     # margin
    batch_size        = 32,
    num_epochs        = 30,
    lr_head           = 1e-3,     # head + projector LR during warmup
    lr_backbone       = 2e-5,     # last 4 ViT blocks LR during fine-tune
    warmup_epochs     = 3,        # freeze backbone for this many epochs first
    mls_threshold     = None,     # auto-calibrated from train MLS distribution
    mls_pct           = 5,        # percentile of train MLS used as threshold
    dbscan_eps        = 0.5,      # cosine distance cutoff for DBSCAN
    dbscan_min_samples= 2,
    device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'),
    seed              = 42,
    unfrozen_blocks   = 4,
    weight_decay  = 5e-4,
)
torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
print('Device:', CFG['device'])

Device: mps


## Data Loading 

In [16]:
CKPT_DIR = os.path.join(ROOT, 'checkpoints/dinov2/')
os.makedirs(CKPT_DIR, exist_ok=True)

In [18]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupShuffleSplit

meta = pd.read_csv(os.path.join(ROOT, 'metadata.csv'))

# All training images with known identity labels
all_train_df = meta[(meta['split'] == 'train') & (meta['identity'].notna())].copy()
query_df     = meta[meta['split'] == 'test'].copy()

# Split by identity group: ~15% of identities go to val (their images are "novel" to the model)
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=CFG['seed'])
train_idx, val_idx = next(gss.split(all_train_df, groups=all_train_df['identity']))
train_meta = all_train_df.iloc[train_idx].reset_index(drop=True)
val_meta   = all_train_df.iloc[val_idx].reset_index(drop=True)

# Fit LabelEncoder on train identities only
le = LabelEncoder()
le.fit(train_meta['identity'].values)

NUM_CLASSES = len(le.classes_)
print(f'Known identities (train classes): {NUM_CLASSES}')
print(f'Train images : {len(train_meta)}')
print(f'Val images   : {len(val_meta)}  ({val_meta["identity"].nunique()} novel identities)')
print(f'Query images : {len(query_df)}')

Known identities (train classes): 936
Train images : 11578
Val images   : 1496  (166 novel identities)
Query images : 2409


In [19]:
class TrainDataset(Dataset):
    """Returns (image_tensor, integer_label) for ArcFace training."""
    def __init__(self, metadata: pd.DataFrame, le: LabelEncoder,
                 transform, root: str):
        self.paths     = metadata['path'].values
        self.labels    = le.transform(metadata['identity'].values)
        self.transform = transform
        self.root      = root
        self.le        = le

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.root, self.paths[idx])).convert('RGB')
        return self.transform(img), self.labels[idx]

    @property
    def num_classes(self):
        return len(self.le.classes_)


class ValDataset(Dataset):
    """
    Returns (image_tensor, identity_string) for open-set evaluation.
    Contains a mix of known individuals (identity in train) and novel
    individuals (identity never seen in training).
    """
    def __init__(self, metadata: pd.DataFrame, transform, root: str):
        self.paths      = metadata['path'].values
        self.identities = metadata['identity'].fillna('unknown').values
        self.transform  = transform
        self.root       = root

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.root, self.paths[idx])).convert('RGB')
        return self.transform(img), self.identities[idx]


size = CFG['img_size']
mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_tf = T.Compose([
    T.Resize((size, size)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    T.ToTensor(),
    T.Normalize(mean, std),
])
val_tf = T.Compose([
    T.Resize((size, size)),
    T.ToTensor(),
    T.Normalize(mean, std),
])

num_workers = 0  # 0 for Colab; set to 4 for local

train_ds  = TrainDataset(train_meta, le, train_tf, ROOT)
val_ds    = ValDataset(val_meta,         val_tf,   ROOT)
query_ds  = ValDataset(query_df,         val_tf,   ROOT)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=num_workers, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=num_workers)
query_loader = DataLoader(query_ds, batch_size=64, shuffle=False, num_workers=num_workers)

print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}  |  Query batches: {len(query_loader)}')

Train batches: 361  |  Val batches: 24  |  Query batches: 38


## Model

In [20]:
class ArcFaceLoss(nn.Module):
    """Additive Angular Margin Loss.
    Adds a fixed margin to the target class angle before softmax,
    pushing intra-class embeddings together and inter-class apart.
    """
    def __init__(self, in_features, num_classes, s=30.0, m=0.50):
        super().__init__()
        self.s, self.m = s, m
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, emb, labels):
        cosine = F.linear(F.normalize(emb), F.normalize(self.weight))  # (B, C)
        sine   = (1.0 - cosine.pow(2)).clamp(0, 1).sqrt()
        phi    = cosine * self.cos_m - sine * self.sin_m  # cos(theta + margin)
        phi    = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine).scatter_(1, labels.view(-1, 1), 1.0)
        logits  = (one_hot * phi + (1.0 - one_hot) * cosine) * self.s
        return F.cross_entropy(logits, labels, label_smoothing=0.1), logits

In [21]:
class DINOv2ReID(nn.Module):
    def __init__(self, model_name, embed_dim, num_classes, s, m):
        super().__init__()
        # num_classes=0, backbone returns raw [CLS] token embedding
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        bb_dim = self.backbone.num_features  # 768 for ViT-Base
        # Optional linear projection; keeps dim if embed_dim == bb_dim
        self.projector = (
            nn.Sequential(nn.Linear(bb_dim, embed_dim), nn.BatchNorm1d(embed_dim))
            if embed_dim != bb_dim else nn.Identity()
        )
        self.arcface = ArcFaceLoss(embed_dim, num_classes, s=s, m=m)

    def embed(self, x):
        """Return L2-normalised embedding (used at inference)."""
        return F.normalize(self.projector(self.backbone(x)), dim=1)

    def forward(self, x, labels=None):
        emb = self.embed(x)
        if labels is not None:
            loss, logits = self.arcface(emb, labels)
            return loss, logits, emb
        return emb

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_last_blocks(self, n=4):
        for p in self.backbone.parameters():
            p.requires_grad = False
        for block in list(self.backbone.blocks)[-n:]:
            for p in block.parameters():
                p.requires_grad = True
        for p in self.backbone.norm.parameters():
            p.requires_grad = True


model = DINOv2ReID(
    model_name=CFG['model_name'],
    embed_dim=CFG['embed_dim'],
    num_classes=NUM_CLASSES,
    s=CFG['arcface_s'],
    m=CFG['arcface_m'],                           
).to(CFG['device'])

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Total params: {total_params:.1f}M')

Total params: 86.7M


## load previous checkpoint